# OrgOS — GRPO Training on a Multi-App Enterprise RL Environment

**Project:** OrgOS — an OpenEnv environment that simulates enterprise workflows across **Jira, Zendesk, Salesforce, and Workday** with realistic challenges: schema drift, RBAC, SLA constraints, and policy drift.

**Goal of this notebook:** Fine-tune `Qwen2.5-3B-Instruct` with **GRPO** (Group Relative Policy Optimization) using **live environment rewards**, then compare the trained agent against the untrained baseline.

**Hardware:** Colab T4 (free tier, 16 GB VRAM). End-to-end runtime ≈ 45–60 min.

**Outputs (committed to the repo):**
- `training/training_log.txt` — structured logs (`[TRAIN_CONFIG]`, `[EVAL]`, `[TRAIN_STEP]`, …)
- `training/plots/training_curve.png` — mean reward vs GRPO step
- `training/plots/baseline_vs_trained.png` — per-workflow comparison
- `training/plots/score_distribution.png` — per-episode score distribution
- `training/orgos_lora_adapter/` — trained LoRA weights

Reviewers can open this notebook on Colab → Runtime → *Run all* and reproduce every artifact end-to-end.

## 1. Setup — install dependencies and clone the repo

In [ ]:
# Pin TRL to the version Unsloth requires BEFORE installing unsloth.
# trl 1.x breaks Unsloth's GRPOTrainer patches — keep it <=0.24.
%pip install -q "trl>=0.18.2,<=0.24.0" peft accelerate bitsandbytes datasets
# Install Unsloth after TRL so its patches apply to the right TRL version.
%pip install -q --upgrade unsloth
%pip install -q fastapi 'uvicorn[standard]' pydantic httpx faker openai aiofiles

In [ ]:
# Clone the OrgOS dev repo (env server, models, business rules)
import os
REPO_URL = 'https://github.com/Tanvi51204/OpenEnv-Round-2.git'
if not os.path.isdir('/content/OpenEnv-Round-2'):
    !git clone {REPO_URL} /content/OpenEnv-Round-2
%cd /content/OpenEnv-Round-2

In [ ]:
# Imports — keep `import unsloth` first to register its patches.
import unsloth

import json, os, re, sys, time, subprocess
from pathlib import Path
from typing import List

import httpx
import numpy as np
import torch
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import TrainerCallback
from trl import GRPOConfig, GRPOTrainer
from unsloth import FastLanguageModel

torch.set_float32_matmul_precision('high')

## 2. Configuration

In [ ]:
# ---- Model ----
MODEL_NAME    = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
MAX_SEQ_LEN   = 4096
LORA_R        = 16
LORA_ALPHA    = 16

# ---- Environment ----
ENV_URL       = 'http://localhost:8000'
WORKFLOWS     = ['A', 'B', 'C']

# ---- Data / eval ----
N_PROMPTS_PER_WORKFLOW = 20      # 20 × 3 = 60 prompts
N_EVAL_EPISODES        = 5       # episodes per workflow at eval time
MAX_EPISODE_STEPS      = 15

# ---- GRPO ----
MAX_TRAIN_STEPS        = 150     # more steps for better convergence
NUM_GENERATIONS        = 2       # G = candidates per prompt (keep low for T4 VRAM)
PER_DEVICE_BATCH       = 1
GRAD_ACCUM             = 2       # effective batch = 2 with grad accum
LEARNING_RATE          = 8e-6
MAX_COMPLETION_LENGTH  = 256
REWARD_STEPS           = 2       # multi-step rollout depth in reward fn

# ---- Drive checkpoint (saves every N steps so Colab disconnects don't lose progress) ----
CKPT_EVERY_STEPS = 30

# ---- Output paths ----
TRAIN_DIR   = Path('/content/OpenEnv-Round-2/training')
PLOTS_DIR   = TRAIN_DIR / 'plots'
ADAPTER_DIR = TRAIN_DIR / 'orgos_lora_adapter'
LOG_PATH    = TRAIN_DIR / 'training_log.txt'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Structured logger — every important event goes through this so submission has a clean log.
LOG_PATH.write_text('')   # truncate

def tlog(line: str):
    print(line, flush=True)
    with open(LOG_PATH, 'a') as f:
        f.write(line + '\n')

## 3. Start the OrgOS environment server

We launch the FastAPI env server (`server/app.py`) as a background subprocess. The reward function and eval loop call it over HTTP at `localhost:8000`.

In [ ]:
ENV_PROC = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'server.app:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd='/content/OpenEnv-Round-2',
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
for _ in range(30):
    try:
        r = httpx.get(f'{ENV_URL}/health', timeout=2)
        if r.status_code == 200:
            tlog(f"[ENV] status={r.json().get('status')} version={r.json().get('version','?')}")
            break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('Env server failed to start')

## 4. Load model — Qwen2.5-3B-Instruct, 4-bit, with LoRA adapters

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    target_modules = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing = 'unsloth',
)

# Clear max_length so generate() doesn't warn about max_new_tokens vs max_length conflict.
model.config.max_length = None

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
tlog(f'[TRAIN_CONFIG] model={MODEL_NAME} lora_r={LORA_R} max_seq_len={MAX_SEQ_LEN} '
     f'trainable_params={trainable:,} quantization=4bit')

## 5. Helpers — system prompt, observation formatting, action parsing

The agent gets a **stateless single-turn prompt**: `[SYSTEM_PROMPT] + [observation]` → `[action JSON]`. This matches what GRPO trains on, which is critical for eval/train alignment, and prevents context accumulation over a multi-step episode.

In [ ]:
SYSTEM_PROMPT = '''You are OrgOS Agent — an enterprise workflow automation agent.
You operate across four SaaS apps: Jira, Zendesk, Salesforce, and Workday.

Each turn you receive a JSON observation with workflow_goal, pending_steps, app_states,
schema_hints (field renames in effect this episode, e.g. {"jira.priority": "severity"}),
active_rules, message (feedback from last action), and current_score.

Respond ONLY with a valid JSON object — no markdown, no explanation:
  {"app": "<app>", "operation": "<op>", "args": {...}}

Available apps and key operations:
  jira:       get_issue, create_issue, update_status, set_priority, assign_owner,
              add_label, link_zendesk_ticket, close_issue, list_issues
  zendesk:    get_ticket, acknowledge_ticket, set_urgency, assign_agent,
              escalate_to_jira, resolve_ticket, add_note, list_tickets, create_agent_profile
  salesforce: get_account, list_accounts, update_deal_stage, flag_churn_risk,
              assign_account_owner, log_interaction, get_opportunity
  workday:    get_employee, list_employees, provision_access, log_sla_event,
              request_budget_approval, create_onboarding_task, complete_task

CRITICAL RULES:
1. Read schema_hints FIRST. If "salesforce.owner" -> "rep_email", use "rep_email" not "owner".
2. Complete pending_steps in order.
3. Never repeat a failed action unchanged — read the message and adapt.
4. Use list_* operations to discover record IDs.
5. Stop when pending_steps is empty or done=true.'''

In [ ]:
WORKFLOW_APPS = {
    'A': {'jira', 'zendesk', 'salesforce', 'workday'},
    'B': {'zendesk', 'salesforce', 'workday'},
    'C': {'jira', 'zendesk', 'salesforce'},
}

def obs_to_text(obs: dict) -> str:
    hints   = obs.get('schema_hints', {})
    pending = obs.get('pending_steps', [])
    lines = [
        f"current_score: {obs['current_score']}",
        f"step_count:    {obs['step_count']}",
        f"workflow_id:   {obs['workflow_id']}",
        '',
        '=== WORKFLOW GOAL ===' , obs['workflow_goal'], '',
        '=== PENDING STEPS ===',
        '\n'.join(f'  - {s}' for s in pending) or '  (all steps complete!)',
        '',
        '=== SCHEMA HINTS (use these field names) ===',
        json.dumps(hints, indent=2) if hints else '  (no drift — use canonical names)',
        '',
        '=== ACTIVE RULES ===',
        json.dumps(obs.get('active_rules', {}), indent=2),
        '',
        '=== LAST MESSAGE ===', obs['message'], '',
        '=== APP STATES ===',
    ]
    relevant = WORKFLOW_APPS.get(obs.get('workflow_id', 'A'),
                                 {'jira','zendesk','salesforce','workday'})
    for app_name, view in obs.get('app_states', {}).items():
        if app_name not in relevant:
            continue
        view_str = str(view)
        if len(view_str) > 600:
            view_str = view_str[:600] + '...[truncated]'
        lines += [f'  [{app_name.upper()}]', f'  {view_str}', '']
    return '\n'.join(lines)

def parse_action(text: str):
    text = re.sub(r'```(?:json)?\s*', '', text.strip()).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r'\{.*\}', text, re.DOTALL)
        if m:
            try: return json.loads(m.group())
            except Exception: return None
    return None

def build_prompt(obs_text: str) -> str:
    msgs = [{'role': 'user', 'content': SYSTEM_PROMPT + '\n\n---\n\n' + obs_text}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

## 6. Episode runner & evaluator

`run_episode_with_model` is **stateless** — each step sends just `[system + current obs]`, no chat history. This (a) keeps prompts under `MAX_SEQ_LEN`, (b) matches the GRPO training format exactly, and (c) avoids context accumulation across multi-step episodes.

In [ ]:
def run_episode_with_model(workflow_id: str, max_steps: int = MAX_EPISODE_STEPS) -> float:
    obs = httpx.post(f'{ENV_URL}/reset', json={'workflow_id': workflow_id}).json()['observation']
    for _ in range(max_steps):
        if obs['done']:
            break
        prompt = build_prompt(obs_to_text(obs))
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens = 256,
                do_sample      = False,
                pad_token_id   = tokenizer.eos_token_id,
            )
        action_str = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:],
                                      skip_special_tokens=True).strip()
        action = parse_action(action_str)
        if action is None:
            break
        result = httpx.post(f'{ENV_URL}/step', json=action).json()
        obs    = result['observation']
        if obs['done']:
            break
    return float(obs.get('current_score', 0.001))

def evaluate(phase: str, n_eval: int = N_EVAL_EPISODES) -> dict:
    scores = {wf: [] for wf in WORKFLOWS}
    tlog(f'[EVAL_START] phase={phase}')
    for wf in WORKFLOWS:
        for ep in range(n_eval):
            s = run_episode_with_model(wf)
            scores[wf].append(s)
            tlog(f'[EVAL] phase={phase} workflow={wf} episode={ep+1} score={s:.4f}')
        m = float(np.mean(scores[wf]))
        tlog(f'[EVAL_WORKFLOW] phase={phase} workflow={wf} '
             f'mean={m:.4f} min={min(scores[wf]):.4f} max={max(scores[wf]):.4f}')
    overall = float(np.mean([s for v in scores.values() for s in v]))
    tlog(f'[EVAL_END] phase={phase} overall_mean={overall:.4f}')
    return scores

## 7. Baseline evaluation — *before* training

This is the untrained Qwen2.5-3B reference point. We will compare against this after GRPO training.

In [ ]:
FastLanguageModel.for_inference(model)
baseline_scores = evaluate(phase='baseline')
baseline_overall = float(np.mean([s for v in baseline_scores.values() for s in v]))

## 8. Build the prompt dataset for GRPO

We collect 60 fresh observations (20 per workflow) by resetting the env. GRPO will sample from this dataset, generate G=2 candidate actions per prompt, score each via the live env, and update the policy from the relative advantages.

In [ ]:
rows = []
for wf in WORKFLOWS:
    for _ in range(N_PROMPTS_PER_WORKFLOW):
        obs = httpx.post(f'{ENV_URL}/reset', json={'workflow_id': wf}).json()['observation']
        rows.append({
            'prompt':      build_prompt(obs_to_text(obs)),
            'workflow_id': wf,
        })
prompt_dataset = Dataset.from_list(rows)
tlog(f'[TRAIN_CONFIG] algorithm=GRPO prompts={len(prompt_dataset)} '
     f'workflows={",".join(WORKFLOWS)} prompts_per_workflow={N_PROMPTS_PER_WORKFLOW}')

tok_len = len(tokenizer(prompt_dataset[0]['prompt']).input_ids)
tlog(f'[PROMPT_DEBUG] first_prompt_tokens={tok_len}')

## 9. Reward function — multi-step live environment rollout

For each candidate completion we:
1. Parse it as a JSON action.
2. Reset the env and apply the action (step 1).
3. Continue `REWARD_STEPS-1` more steps with the current model (greedy), accumulating env transitions.
4. Return the **cumulative episode score** — not just single-step reward.

This multi-step signal prevents the model from collapsing to always outputting `list_tickets` (which gives a small single-step reward but doesn't advance the workflow). Invalid JSON gets a −0.1 penalty.

In [ ]:
def orgos_reward_fn(completions: List[str], prompts: List[str] = None, **kwargs) -> List[float]:
    workflow_ids = kwargs.get('workflow_id', ['A'] * len(completions))
    rewards = []
    for completion, wf_id in zip(completions, workflow_ids):
        action = parse_action(completion)
        if action is None:
            rewards.append(-0.1)
            continue
        try:
            # Reset env and apply GRPO-generated action (step 1)
            obs = httpx.post(f'{ENV_URL}/reset', json={'workflow_id': wf_id}, timeout=10).json()['observation']
            result = httpx.post(f'{ENV_URL}/step', json=action, timeout=10).json()
            obs = result['observation']

            # Continue REWARD_STEPS-1 more steps with the current model (greedy)
            for _ in range(REWARD_STEPS - 1):
                if obs.get('done'):
                    break
                prompt = build_prompt(obs_to_text(obs))
                inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
                with torch.no_grad():
                    out = model.generate(
                        **inputs,
                        max_new_tokens = 128,
                        do_sample      = False,
                        pad_token_id   = tokenizer.eos_token_id,
                    )
                cont_str = tokenizer.decode(
                    out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
                ).strip()
                cont_action = parse_action(cont_str)
                if cont_action is None:
                    break
                result = httpx.post(f'{ENV_URL}/step', json=cont_action, timeout=10).json()
                obs    = result['observation']

            # Return cumulative score — rewards multi-step progress, not just single actions
            rewards.append(float(obs.get('current_score', 0.001)))
        except Exception as e:
            rewards.append(-0.1)
    return rewards

# Sanity check
_v = orgos_reward_fn(['{\'app\':\'zendesk\',\'operation\':\'list_tickets\',\'args\':{}}'], workflow_id=['A'])
_i = orgos_reward_fn(['not json'], workflow_id=['A'])
tlog(f'[REWARD_FN_CHECK] valid_action={_v[0]:.4f} invalid_action={_i[0]:.4f}')

## 10. GRPO training

We log every training step's reward into `[TRAIN_STEP]` lines so we can plot a meaningful learning curve.
A Drive checkpoint callback saves the adapter every 30 steps so a Colab disconnect doesn't lose progress.

In [ ]:
training_step_rewards = []   # captured by callback for the plot

class OrgOSLogCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        step   = state.global_step
        reward = logs.get('reward') or logs.get('rewards/orgos_reward_fn') or logs.get('reward/mean')
        loss   = logs.get('loss')
        kl     = logs.get('kl')
        if reward is not None:
            training_step_rewards.append((step, float(reward)))
            tlog(f'[TRAIN_STEP] step={step} reward={float(reward):.4f} '
                 f"loss={('%.4f'%loss) if loss is not None else 'NA'} "
                 f"kl={('%.4f'%kl) if kl is not None else 'NA'}")

    def on_step_end(self, args, state, control, **kwargs):
        """Save checkpoint to Drive every CKPT_EVERY_STEPS steps."""
        if state.global_step % CKPT_EVERY_STEPS == 0 and state.global_step > 0:
            try:
                from google.colab import drive
                drive.mount('/content/drive', force_remount=False)
                ckpt_dir = Path(f'/content/drive/MyDrive/orgos-training/ckpt_step{state.global_step}')
                ckpt_dir.mkdir(parents=True, exist_ok=True)
                model.save_pretrained(str(ckpt_dir))
                import shutil
                shutil.copy(LOG_PATH, ckpt_dir / 'training_log.txt')
                tlog(f'[CHECKPOINT] step={state.global_step} saved to {ckpt_dir}')
            except Exception:
                pass  # not on Colab or Drive not mounted — skip silently

In [ ]:
FastLanguageModel.for_training(model)

# GRPOConfig — using TRL <=0.24 (pinned in cell 2) so max_new_tokens is accepted.
# Unsloth patches this config; max_prompt_length / max_completion_length are NOT supported.
grpo_config = GRPOConfig(
    output_dir                  = '/content/grpo_ckpt',
    num_train_epochs            = 1,
    max_steps                   = MAX_TRAIN_STEPS,
    per_device_train_batch_size = PER_DEVICE_BATCH,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LEARNING_RATE,
    num_generations             = NUM_GENERATIONS,
    max_new_tokens              = MAX_COMPLETION_LENGTH,
    temperature                 = 0.9,
    logging_steps               = 1,
    save_strategy               = 'no',
    report_to                   = 'none',
    bf16                        = False,
    fp16                        = True,
    optim                       = 'adamw_8bit',
)

trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    reward_funcs     = [orgos_reward_fn],
    train_dataset    = prompt_dataset,
    args             = grpo_config,
    callbacks        = [OrgOSLogCallback()],
)

tlog(f'[TRAIN_START] max_steps={MAX_TRAIN_STEPS} G={NUM_GENERATIONS} lr={LEARNING_RATE} reward_steps={REWARD_STEPS}')
trainer.train()
tlog(f'[TRAIN_END] steps_completed={trainer.state.global_step}')

## 11. Post-training evaluation

Same protocol as the baseline (3 workflows × 5 episodes), so the comparison is apples-to-apples.

In [ ]:
FastLanguageModel.for_inference(model)
trained_scores = evaluate(phase='trained')
trained_overall = float(np.mean([s for v in trained_scores.values() for s in v]))

tlog('[TRAIN_SUMMARY] '
     f'baseline_overall={baseline_overall:.4f} trained_overall={trained_overall:.4f} '
     f'delta={trained_overall - baseline_overall:+.4f}')

## 12. Plots

All plots are saved to `training/plots/` and committed to the repo so reviewers can see them in the README.

In [ ]:
# 12a. Training curve — mean reward vs GRPO step
if training_step_rewards:
    steps, rewards = zip(*training_step_rewards)
    plt.figure(figsize=(8,5))
    plt.plot(steps, rewards, marker='o', markersize=3, linewidth=1.5, color='tab:blue', label='per-step reward')
    if len(rewards) >= 5:
        win = max(3, len(rewards)//10)
        roll = np.convolve(rewards, np.ones(win)/win, mode='valid')
        plt.plot(steps[win-1:], roll, color='tab:orange', linewidth=2.5, label=f'rolling mean (w={win})')
    plt.xlabel('GRPO training step')
    plt.ylabel('mean reward (per batch)')
    plt.title('OrgOS GRPO training curve — Qwen2.5-3B-Instruct')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'training_curve.png', dpi=150)
    plt.show()
    tlog('[ARTIFACT] training_curve.png saved')

In [ ]:
# 12b. Baseline vs trained — grouped bar per workflow
x = np.arange(len(WORKFLOWS))
width = 0.35
baseline_means = [np.mean(baseline_scores[wf]) for wf in WORKFLOWS]
trained_means  = [np.mean(trained_scores[wf])  for wf in WORKFLOWS]

fig, ax = plt.subplots(figsize=(8,5))
ax.bar(x - width/2, baseline_means, width, label='baseline (untrained)', color='tab:gray')
ax.bar(x + width/2, trained_means,  width, label='GRPO-trained',         color='tab:green')
ax.set_xticks(x)
ax.set_xticklabels([f'Workflow {wf}' for wf in WORKFLOWS])
ax.set_ylabel('mean episode score (0–1)')
ax.set_ylim(0, 1)
ax.set_title(f'Baseline vs GRPO-trained — overall {baseline_overall:.3f} → {trained_overall:.3f}')
ax.legend()
ax.grid(axis='y', alpha=0.3)
for i, (b, t) in enumerate(zip(baseline_means, trained_means)):
    ax.text(i - width/2, b + 0.01, f'{b:.2f}', ha='center', fontsize=9)
    ax.text(i + width/2, t + 0.01, f'{t:.2f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'baseline_vs_trained.png', dpi=150)
plt.show()
tlog('[ARTIFACT] baseline_vs_trained.png saved')

In [ ]:
# 12c. Per-episode score distribution (box plot)
fig, ax = plt.subplots(figsize=(9,5))
data, labels, colors = [], [], []
for wf in WORKFLOWS:
    data += [baseline_scores[wf], trained_scores[wf]]
    labels += [f'{wf} (base)', f'{wf} (trained)']
    colors += ['lightgray', 'lightgreen']
bp = ax.boxplot(data, labels=labels, patch_artist=True)
for patch, c in zip(bp['boxes'], colors):
    patch.set_facecolor(c)
ax.set_ylabel('episode score (0–1)')
ax.set_title('Per-episode score distribution — baseline vs GRPO-trained')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'score_distribution.png', dpi=150)
plt.show()
tlog('[ARTIFACT] score_distribution.png saved')

## 13. Save artifacts

Saves the LoRA adapter and copies all artifacts to Google Drive so they survive a Colab disconnect.

In [ ]:
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
tlog(f'[ARTIFACT] lora_adapter saved to {ADAPTER_DIR}')

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_DIR = Path('/content/drive/MyDrive/orgos-training')
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    !cp    {LOG_PATH}    {DRIVE_DIR}/
    !cp -r {PLOTS_DIR}   {DRIVE_DIR}/
    !cp -r {ADAPTER_DIR} {DRIVE_DIR}/
    print(f'Artifacts copied to {DRIVE_DIR}')
except ImportError:
    print('Not on Colab — skipping Drive copy')

In [ ]:
# Stop the env server cleanly
ENV_PROC.terminate()
tlog('[RUN_END]')
print('\nDone. Commit these to the repo:')
print(f'  - {LOG_PATH}')
print(f'  - {PLOTS_DIR}/*.png')
print(f'  - {ADAPTER_DIR}/')